# 06. 베이지안 MMM과 ROI

마케팅 업계의 표준 방식인 **베이지안 마케팅 믹스 모델(MMM)**입니다. 선형 회귀와 같은 요인을 쓰지만:

1. **효과를 범위로 추정**: "프로모션 +269개" 대신 "**+249~+286개일 확률 90%**" (90% 신용구간)
2. **상식을 사전분포로 반영**: 할인·프로모션·광고·공휴일 효과는 0 이상, 결품은 0 이하로만 추정
3. **광고 이월률과 포화 정도를 모델이 직접 학습** → 광고비 반응곡선과 ROI 계산에 활용

도구: [PyMC](https://www.pymc.io). 실무 패키지로는 PyMC-Marketing, Google Meridian, Meta Robyn이 있습니다.

In [ ]:
import sys, warnings
sys.path.append("../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 한글 글꼴
from common import *
plt.rcParams["figure.dpi"] = 110
df = load_data()
y = df.sellout.values.astype(float)
print(df.shape)

## 모델 구조

```
셀아웃 ~ Normal(μ, σ)
μ = 기본수요 + 계절성 + β_할인·할인율 + β_프로모션·프로모션
  + β_광고 · 포화( adstock(광고비, decay), lam )
  + β_공휴일·공휴일 + β_기온·(기온−평균) − β_결품·결품일수

adstock: 이번 주 + decay × 지난주 + decay² × 2주 전 + …  (최대 8주)
포화(x) = (1 − e^(−lam·x)) / (1 + e^(−lam·x))   ← 0에서 1로 점점 완만하게 올라가는 곡선
```

**사전분포**(데이터를 보기 전의 상식):
- β_할인, β_프로모션, β_광고, β_공휴일, β_결품 ~ HalfNormal → **0 이상만 가능** (결품은 빼기로 넣어 음수 효과)
- decay ~ Beta(2, 2) → 0~1 사이, 중간값 근처를 약하게 선호
- 나머지는 넓은 Normal (거의 정보 없음)

In [ ]:
import pymc as pm
import pytensor.tensor as pt
import arviz as az

L_MAX = 8
temp_mean, ad_scale = df.temp.mean(), df.ad_spend.max()
t = np.arange(len(df), dtype=float)

def lag_matrix(x, L):
    M = np.zeros((len(x), L))
    for l in range(L):
        M[l:, l] = x[: len(x) - l]
    return M

ad_lag = lag_matrix(df.ad_spend.values.astype(float), L_MAX) / ad_scale

with pm.Model() as mmm:
    intercept = pm.Normal("intercept", 900, 300)
    trend = pm.Normal("trend", 0, 5)
    s1, c1 = pm.Normal("s1", 0, 150), pm.Normal("c1", 0, 150)
    b_disc = pm.HalfNormal("b_disc", 30)
    b_promo = pm.HalfNormal("b_promo", 400)
    b_ad = pm.HalfNormal("b_ad", 400)
    decay = pm.Beta("decay", 2, 2)
    lam = pm.Gamma("lam", 2, 1)
    b_hol = pm.HalfNormal("b_hol", 300)
    b_temp = pm.Normal("b_temp", 0, 20)
    b_stock = pm.HalfNormal("b_stock_abs", 200)
    sigma = pm.HalfNormal("sigma", 100)

    adst = pt.dot(ad_lag, decay ** pt.arange(L_MAX))
    sat = (1 - pt.exp(-lam * adst)) / (1 + pt.exp(-lam * adst))
    comps = {
        "기본 수요": intercept + trend * t,
        "계절성": s1 * np.sin(2*np.pi*t/52) + c1 * np.cos(2*np.pi*t/52),
        "할인": b_disc * df.discount_pct.values, "프로모션": b_promo * df.promo.values,
        "광고": b_ad * sat, "공휴일": b_hol * df.holiday.values,
        "기온": b_temp * (df.temp.values - temp_mean), "결품": -b_stock * df.stockout_days.values,
    }
    for k, v in comps.items():
        pm.Deterministic(k, v)
    mu = pm.Deterministic("mu", sum(comps.values()))
    pm.Normal("obs", mu, sigma, observed=y)
    idata = pm.sample(1000, tune=1000, chains=4, target_accept=0.92, random_seed=11, progressbar=False)

## 수렴 진단

베이지안 모델은 샘플링으로 답을 찾기 때문에, 제대로 수렴했는지 먼저 확인합니다.
- **R-hat ≤ 1.01**: 여러 체인이 같은 답에 도달했다
- **발산(divergence) 0회**: 샘플링에 문제가 없었다

In [ ]:
summ = az.summary(idata, var_names=["b_disc", "b_promo", "b_ad", "decay", "lam", "b_hol", "b_temp", "b_stock_abs", "sigma"], ci_prob=0.9, ci_kind="eti")
print("R-hat 최대:", summ.r_hat.max(), "/ 발산:", int(idata.sample_stats.diverging.sum()))
summ.filter(regex="mean|_lb|_ub|r_hat").round(3)

## 추정치 vs 정답 (90% 범위)

In [ ]:
post = idata.posterior
post = post.to_dataset() if hasattr(post, "to_dataset") else post
P = post.stack(s=("chain", "draw"))
q = lambda a: (np.mean(a), np.percentile(a, 5), np.percentile(a, 95))
est = {
    "할인 (1%p당)": (q(P.b_disc.values), 14), "프로모션": (q(P.b_promo.values), 260),
    "공휴일": (q(P.b_hol.values), 180), "기온 (1°C당)": (q(P.b_temp.values), -6),
    "결품 (1일당)": (q(-P.b_stock_abs.values), -95), "광고 이월률": (q(P.decay.values), 0.5),
}
fig, axes = plt.subplots(1, len(est), figsize=(14, 2.6))
for ax, (k, ((m, lo, hi), tv)) in zip(axes, est.items()):
    ax.errorbar([0], [m], yerr=[[m - lo], [hi - m]], fmt="o", color="#2f6bd4", capsize=6)
    ax.axhline(tv, color="black", ls="--", lw=1)
    ax.set_title(k, fontsize=9); ax.set_xticks([])
    ax.text(0.05, tv, " 정답", va="bottom", fontsize=8)
plt.suptitle("파란 점: 평균, 막대: 90% 범위, 점선: 정답", fontsize=9); plt.tight_layout()
pd.DataFrame({k: dict(평균=round(m, 2), 하한=round(lo, 2), 상한=round(hi, 2), 정답=tv, 범위안=lo <= tv <= hi)
              for k, ((m, lo, hi), tv) in est.items()}).T

6개 중 5~6개가 90% 범위 안에 들어옵니다. 광고 이월률은 정답(50%)보다 약간 높게 잡히는데, 광고는 효과를 분리하기 가장 어려운 요인입니다.

## 한 주의 기여도를 범위와 함께

In [ ]:
i = 29
rows = {}
for k in FACTORS:
    a = P[k].values[i]
    rows[k] = dict(평균=a.mean(), 하한=np.percentile(a, 5), 상한=np.percentile(a, 95))
mu_i = P["mu"].values[i]
rows["모델 예측"] = dict(평균=mu_i.mean(), 하한=np.percentile(mu_i, 5), 상한=np.percentile(mu_i, 95))
print(f"{df.week[i]:%Y-%m-%d} 주, 실제 판매 {y[i]:.0f}개")
pd.DataFrame(rows).T.round(0)

## 광고비 반응곡선 (포화 효과)

매주 같은 금액을 꾸준히 쓴다고 가정했을 때의 주간 판매 증가량입니다. 곡선이 완만해질수록 **추가 지출의 효율이 떨어집니다**.

In [ ]:
spend = np.linspace(0, 60, 61)
geo = np.array([P.decay.values ** l for l in range(L_MAX)]).sum(axis=0)
x = spend[:, None] * geo[None, :] / ad_scale
resp = P.b_ad.values[None, :] * (1 - np.exp(-P.lam.values * x)) / (1 + np.exp(-P.lam.values * x))
fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(spend, np.percentile(resp, 5, axis=1), np.percentile(resp, 95, axis=1), color="#2f6bd4", alpha=.2, label="90% 범위")
ax.plot(spend, resp.mean(axis=1), color="#2f6bd4", lw=2.5, label="평균")
avg = df.ad_spend.mean()
ax.axvline(avg, color="black", lw=1); ax.text(avg + .5, 5, f"현재 평균 {avg:.1f}", fontsize=9)
ax.axvline(df.ad_spend.max(), color="grey", ls="--"); ax.text(df.ad_spend.max() + .5, 5, "과거 최대 (이후는 외삽)", fontsize=9)
ax.set_xlabel("주간 광고비 (백만원)"); ax.set_ylabel("판매 증가 (개/주)"); ax.legend(); ax.set_title("광고비 반응곡선")

def resp_at(s):
    xx = s * geo / ad_scale
    return P.b_ad.values * (1 - np.exp(-P.lam.values * xx)) / (1 + np.exp(-P.lam.values * xx))
avg_eff = resp_at(avg) / avg
marg = resp_at(avg + 1) - resp_at(avg)
print(f"지금까지 1백만원당 평균 {avg_eff.mean():.1f}개 ({np.percentile(avg_eff, 5):.1f}~{np.percentile(avg_eff, 95):.1f})")
print(f"여기서 1백만원 더 쓰면 추가 {marg.mean():.1f}개 ({np.percentile(marg, 5):.1f}~{np.percentile(marg, 95):.1f})  ← 한계 효과")

## ROI 계산

```
ROI = (늘어난 판매로 번 이익 − 쓴 비용) ÷ 쓴 비용       0%면 본전, 플러스면 남는 장사
```

**예시 가정값** (실제 숫자로 바꿔서 다시 실행하세요)
- 판매가 200,000원, 개당 이익(공헌이익) 60,000원
- 프로모션 1회 비용 8,000,000원 (진열비·판촉물·인건비)
- 할인 비용 = 할인율 × 판매가 × **그 주 전체 판매량** ← 원래 정가에 사려던 고객에게도 깎아주므로
- 광고 비용 = 광고비

In [ ]:
PRICE, MARGIN, PROMO_COST = 200_000, 60_000, 8_000_000

def roi_row(name, units_draws, cost):
    r = (units_draws * MARGIN - cost) / cost
    lo, hi = np.percentile(r, 5), np.percentile(r, 95)
    verdict = "남는 장사" if lo > 0 else ("손해" if hi < 0 else "불확실")
    return dict(활동=name, 비용_만원=round(cost / 1e4), 늘어난판매=round(units_draws.mean()),
                ROI=f"{r.mean():+.0%}", 범위=f"{lo:+.0%} ~ {hi:+.0%}", 판정=verdict)

tot = lambda k: P[k].values.sum(axis=0)   # 2년 합계 (표본별)
disc_cost = (df.discount_pct / 100 * PRICE * df.sellout).sum()
roi = pd.DataFrame([
    roi_row("할인", tot("할인"), disc_cost),
    roi_row("프로모션", tot("프로모션"), PROMO_COST * df.promo.sum()),
    roi_row("광고", tot("광고"), df.ad_spend.sum() * 1e6),
])
roi

In [ ]:
marg_roi = (marg * MARGIN - 1e6) / 1e6
print(f"광고 1백만원 추가 시 ROI: {marg_roi.mean():+.0%} ({np.percentile(marg_roi, 5):+.0%} ~ {np.percentile(marg_roi, 95):+.0%})")
gain = P.b_disc.values.mean() * MARGIN
cost = 0.01 * PRICE * df.sellout.mean()
print(f"할인 1%p: 이익 +{gain/1e4:.0f}만원 vs 비용 {cost/1e4:.0f}만원 (주당)")
print(f"결품으로 2년간 놓친 이익: 약 {abs(tot('결품').mean()) * MARGIN / 1e8:.1f}억원")

## 결과 해석 (예시 가정값 기준)
- **프로모션**: ROI 약 +100% → 쓴 돈의 약 2배를 벌었습니다.
- **광고**: 평균은 본전 근처지만 범위가 0을 걸쳐 **불확실**. 현재 수준에서 **증액은 비추천**(한계 ROI 약 −65%).
- **할인**: 확실한 손해. 판매량은 늘지만 원래 사려던 고객에게도 깎아주는 비용이 더 큽니다.
- **결품**: 2년간 약 1.4억원의 이익을 놓쳤습니다.

> 신용구간은 **모델 구조가 맞다는 전제** 아래의 범위입니다. 빠진 요인이 있거나 프로모션이 성수기에만 몰렸다면 범위 자체가 틀릴 수 있습니다.

인터랙티브 버전(가정값 입력 가능): `dashboards/02_bayesian_mmm_roi.html`